CELL 1

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

df_app = pd.read_csv('../data/processed/fact_applications.csv')
df_bureau = pd.read_csv('../data/processed/dim_bureau_history.csv')
df_prev = pd.read_csv('../data/processed/dim_prev_applications.csv')

df_stat = (
    df_app
    .merge(df_bureau, on='sk_id_curr', how='left')
    .merge(df_prev, on='sk_id_curr', how='left')
)

for col in ['bureau_total_loans', 'bureau_active_loans', 'bureau_has_overdue', 
            'prev_app_count', 'prev_refused_count', 'prev_refusal_rate']:
    df_stat[col] = df_stat[col].fillna(0)

df_stat['prior_refusal_flag'] = np.where(df_stat['prev_refused_count'] > 0, 'Has Prior Refusal', 'No Prior Refusal')
df_stat['external_overdue_flag'] = np.where(df_stat['bureau_has_overdue'] > 0, 'Has Overdue', 'No Overdue')

print(f"Dataset siap untuk pengujian statistik: {df_stat.shape[0]:,} observasi (alpha = 0.05)")

Dataset siap untuk pengujian statistik: 307,504 observasi (alpha = 0.05)


CELL 2

In [ ]:
cat_vars = [
    'ext_score_tier', 'age_group', 'name_education_type', 'name_income_type',
    'prior_refusal_flag', 'name_contract_type', 'dti_tier', 'external_overdue_flag'
]

chi2_results = []
n_total = len(df_stat)

for col in cat_vars:
    contingency = pd.crosstab(df_stat[col], df_stat['target'])
    chi2, p_val, dof, _ = stats.chi2_contingency(contingency)
    
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n_total * min_dim))
    
    chi2_results.append({
        'Variabel_Kategorikal': col,
        'Chi2_Statistic': round(chi2, 2),
        'DoF': dof,
        'p_value': f"{p_val:.2e}" if p_val < 0.0001 else round(p_val, 4),
        'Cramers_V': round(cramers_v, 4),
        'Signifikan (alpha=0.05)': 'Ya (Tolak H0)' if p_val < 0.05 else 'Tidak'
    })

df_chi2_summary = pd.DataFrame(chi2_results).sort_values(by='Cramers_V', ascending=False)
print("=== HASIL UJI CHI-SQUARE OF INDEPENDENCE (DIURUTKAN DARI KEKUATAN ASOSIASI) ===")
display(df_chi2_summary)

=== HASIL UJI CHI-SQUARE OF INDEPENDENCE (DIURUTKAN DARI KEKUATAN ASOSIASI) ===


,Variabel_Kategorikal,Chi2_Statistic,DoF,p_value,Cramers_V,Signifikan (alpha=0.05)
0,ext_score_tier,"14,024.8000",2,0.00e+00,0.2136,Ya (Tolak H0)
1,age_group,"1,807.6100",4,0.00e+00,0.0767,Ya (Tolak H0)
3,name_income_type,"1,253.1900",7,2.22e-266,0.0638,Ya (Tolak H0)
2,name_education_type,"1,018.9900",4,2.74e-219,0.0576,Ya (Tolak H0)
4,prior_refusal_flag,"1,014.2300",1,1.45e-222,0.0574,Ya (Tolak H0)
5,name_contract_type,292.9100,1,1.15e-65,0.0309,Ya (Tolak H0)
7,external_overdue_flag,282.5100,1,2.14e-63,0.0303,Ya (Tolak H0)
6,dti_tier,117.7000,3,2.41e-25,0.0196,Ya (Tolak H0)


CELL 3

In [3]:
num_vars = [
    'avg_ext_source', 'age_years', 'employment_years', 'prev_refusal_rate',
    'bureau_active_loans', 'annuity_to_income_ratio', 'amt_income_total', 'amt_credit'
]

mw_results = []

for col in num_vars:
    # Pisahkan kelompok lancar (0) dan gagal bayar (1), abaikan NaN (misal employment_years pensiunan)
    group_0 = df_stat[df_stat['target'] == 0][col].dropna()
    group_1 = df_stat[df_stat['target'] == 1][col].dropna()
    
    u_stat, p_val = stats.mannwhitneyu(group_0, group_1, alternative='two-sided')
    med_0 = group_0.median()
    med_1 = group_1.median()
    
    mw_results.append({
        'Variabel_Numerik': col,
        'Median_Lancar (0)': round(med_0, 4),
        'Median_Default (1)': round(med_1, 4),
        'Selisih_Median': round(med_1 - med_0, 4),
        'MannWhitney_U': round(u_stat, 0),
        'p_value': f"{p_val:.2e}" if p_val < 0.0001 else round(p_val, 4),
        'Signifikan (alpha=0.05)': 'Ya (Tolak H0)' if p_val < 0.05 else 'Tidak'
    })

df_mw_summary = pd.DataFrame(mw_results)
print("=== HASIL UJI MANN-WHITNEY U (PERBANDINGAN MEDIAN LANCAR VS DEFAULT) ===")
display(df_mw_summary)

=== HASIL UJI MANN-WHITNEY U (PERBANDINGAN MEDIAN LANCAR VS DEFAULT) ===


,Variabel_Numerik,Median_Lancar (0),Median_Default (1),Selisih_Median,MannWhitney_U,p_value,Signifikan (alpha=0.05)
0,avg_ext_source,0.5339,0.4125,-0.1214,"5,042,708,104.0000",0.00e+00,Ya (Tolak H0)
1,age_years,43.5000,39.1000,-4.4000,"4,090,999,154.0000",0.00e+00,Ya (Tolak H0)
2,employment_years,4.6000,3.4000,-1.2000,"2,928,559,672.0000",0.00e+00,Ya (Tolak H0)
3,prev_refusal_rate,0.0000,0.0000,0.0000,"3,088,966,056.0000",6.42e-309,Ya (Tolak H0)
4,bureau_active_loans,1.0000,2.0000,1.0000,"3,302,856,947.0000",9.25e-56,Ya (Tolak H0)
5,annuity_to_income_ratio,0.1623,0.1693,0.0070,"3,371,452,288.0000",1.47e-24,Ya (Tolak H0)
6,amt_income_total,"148,500.0000","135,000.0000","-13,500.0000","3,643,032,022.0000",9.47e-24,Ya (Tolak H0)
7,amt_credit,"517,788.0000","497,520.0000","-20,268.0000","3,639,041,913.0000",2.33e-22,Ya (Tolak H0)


CELL 4

In [4]:
top_income_types = ['Working', 'Commercial associate', 'Pensioner', 'State servant']
df_kw = df_stat[df_stat['name_income_type'].isin(top_income_types)]

kw_results = []
for metric in ['avg_ext_source', 'annuity_to_income_ratio', 'amt_credit']:
    groups = [group[metric].dropna().values for _, group in df_kw.groupby('name_income_type')]
    h_stat, p_val = stats.kruskal(*groups)
    
    kw_results.append({
        'Metrik_Diuji': metric,
        'Kruskal_H_Stat': round(h_stat, 2),
        'p_value': f"{p_val:.2e}" if p_val < 0.0001 else round(p_val, 4),
        'Kesimpulan': 'Terdapat perbedaan signifikan antar jenis pendapatan' if p_val < 0.05 else 'Tidak berbeda signifikan'
    })

print("=== HASIL UJI KRUSKAL-WALLIS H (LINTAS 4 JENIS PENDAPATAN UTAMA) ===")
display(pd.DataFrame(kw_results))

print("\nRingkasan Median per Jenis Pendapatan:")
display(df_kw.groupby('name_income_type')[['avg_ext_source', 'annuity_to_income_ratio', 'amt_credit']].median().round(4))

=== HASIL UJI KRUSKAL-WALLIS H (LINTAS 4 JENIS PENDAPATAN UTAMA) ===


,Metrik_Diuji,Kruskal_H_Stat,p_value,Kesimpulan
0,avg_ext_source,"7,006.5100",0.00e+00,Terdapat perbedaan signifikan antar jenis pend...
1,annuity_to_income_ratio,"2,348.1700",0.00e+00,Terdapat perbedaan signifikan antar jenis pend...
2,amt_credit,"4,014.5700",0.00e+00,Terdapat perbedaan signifikan antar jenis pend...



Ringkasan Median per Jenis Pendapatan:


,avg_ext_source,annuity_to_income_ratio,amt_credit
name_income_type,,,
Commercial associate,0.5328,0.1524,"553,806.0000"
Pensioner,0.5666,0.1761,"457,515.0000"
State servant,0.5396,0.1625,"558,000.0000"
Working,0.5073,0.1633,"497,520.0000"
